In [ ]:
include( "../geom.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 1024

# Import select speeds.
include( "../0-params/spd-list.jl" )
slist = slist[12:15]
Ns = length( slist )

# Create folder and extract critical noise.
folderdata = [createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) )
    for ρ ∈ ρlist, s ∈ slist]

# Extract critical noise values.
ηcdata = first.( readdlm.( folderdata.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Importing data."; dt=1.0 )

# Initialize data variables.
Adata = [Vector{Dict{defInt,Vector{defFloat}}}( undef, Ns ) for _ ∈ 1:Nρ]
ℓdata = [Vector{Dict{defInt,Vector{defFloat}}}( undef, Ns ) for _ ∈ 1:Nρ]

# Import cluster dynamical data.
for i ∈ 1:Nρ, j ∈ 1:Ns
    Adata[i][j] = loaddict( folderdata[i,j]*"convex-hull-area.txt", defFloat )
    ℓdata[i][j] = loaddict( folderdata[i,j]*"convex-hull-perimeter.txt", defFloat )
    next!( meter )
end

In [ ]:
# Compute mean and standard deviation.
ιmin = 2
ιdatalist = [[[ι for ι in ιmin:length( Adata[i][j] ) if !isempty( Adata[i][j][ι] )]
    for j ∈ 1:Ns] for i ∈ 1:Nρ]
ιmax = round( log10( N ) )

# Area.
μAdatalist = [[[mean( Adata[i][j][ι] ) for ι ∈ ιdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]
σAdatalist = [[[std(  Adata[i][j][ι] ) for ι ∈ ιdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]

# Perimeter.
μℓdatalist = [[[mean( ℓdata[i][j][ι] ) for ι ∈ ιdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]
σℓdatalist = [[[std(  ℓdata[i][j][ι] ) for ι ∈ ιdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ];

In [ ]:
# Extract the x and y-axis variables for fitting (area).
xAdata = [[vcat( [fill( ι, length( Adata[i][j][ι] ) ) for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ]
yAdata = [[vcat( [Adata[i][j][ι] for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ]

# Perimeter.
xℓdata = [[vcat( [fill( ι, length( ℓdata[i][j][ι] ) ) for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ]
yℓdata = [[vcat( [ℓdata[i][j][ι] for ι in ιdatalist[i][j]]...) for j ∈ 1:Ns] for i ∈ 1:Nρ];

In [ ]:
ℓdata[1][1][ιdatalist[1][1][4]]

In [ ]:
# Remove any zero values.
κdatalist = [[(0.0 .< yAdata[i][j]) .& (0.0 .< yℓdata[i][j]) for j ∈ 1:Ns] for i ∈ 1:Nρ]
xAdata = [[xAdata[i][j][κdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]
yAdata = [[yAdata[i][j][κdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]
xℓdata = [[xℓdata[i][j][κdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ]
yℓdata = [[yℓdata[i][j][κdatalist[i][j]] for j ∈ 1:Ns] for i ∈ 1:Nρ];

In [ ]:
# Perform power law fits.
βℓdata = [[fitlogpower( xℓdata[i][j], yℓdata[i][j]; cut=2 ) for j ∈ 1:Ns] for i ∈ 1:Nρ]
βAdata = [[(yℓdata[i][j].^2)\yAdata[i][j] for j ∈ 1:Ns] for i ∈ 1:Nρ]

println( "power law for area: ", βℓdata )
println( "coefficient for perimeter: ", (1/(4π), βAdata) );

In [ ]:
i = 21;  j = 1

# Plot the trend against the mean of the sizes.
plt = plot(; size=(400,200), legend=false )

scatter!( plt, ιdatalist[i][j], μℓdatalist[i][j]; color=:gray )
plot!( plt, 1:N, logpowerlaw.( 1:N; β=βℓdata[i][j] ); color=:cornflowerblue, lw=2 )

plot!( plt; xlims=(1,N), xscale=:log10 )
plot!( plt; ylims=10.0.^(-1,2), yscale=:log10 )
plot!( plt; title="perimeter", titleposition=:left )

In [ ]:
i = 11;  j = 1

# Plot the trend against the mean of the sizes.
plt = plot(; size=(400,200), legend=false )

scatter!( plt, ιdatalist[i][j], μAdatalist[i][j]; color=:gray )
plot!( plt, 1:N, (1/4π).*logpowerlaw.( 1:N; β=βℓdata[i][j] ).^2; color=:cornflowerblue, lw=2 )

plot!( plt; xlims=(1,N), xscale=:log10 )
plot!( plt; ylims=10.0.^(-2,3), yscale=:log10 )
plot!( plt; title="area", titleposition=:left )

In [ ]:
# Save parameter data.
if false
    # Initialize progress meter.
    meter = Progress( Nρ*Ns, "Saving geometric data." )

    for i ∈ 1:Nρ, j ∈ 1:Ns
        # Save the cluster change parameters for each parameter case.
        writedlm( folderdata[i,j]*"mean-convex-hull-area.txt", [μAdatalist[i][j] σAdatalist[i][j]] )
        writedlm( folderdata[i,j]*"mean-convex-hull-perimeter.txt", [μℓdatalist[i][j] σℓdatalist[i][j]] )
        writedlm( folderdata[i,j]*"area-perimeter-scaling.txt", [βℓdata[i][j]; βAdata[i][j]] )

        # Update progress meter.
        next!( meter )
    end
end